# NB00 — Data Preparation

Membersihkan dan menggabungkan tabel mentah `ispu` dan `ispu_weather` menjadi data **per jam per sensor**.

**Input:** `data/raw/ispu.csv`, `data/raw/ispu_weather.csv`
**Output:** `data/processed/hourly_per_sensor.csv`

Setiap cell diberi kode `[NB00-Cxx]` dan diakhiri dengan `✓ [NB00-Cxx] selesai`.
Jika terjadi error, sebutkan kode cell-nya.

> **Catatan zona waktu:** semua waktu di notebook ini masih memakai **jam database** (kolom `datetime_db`).
> Kedua tabel memakai jam yang sama, sehingga merge dan agregasi per jam tidak terpengaruh.
> Konversi ke WIT dilakukan di NB03.

In [1]:
# ============================================================
# [NB00-C01] SETUP & KONFIGURASI
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ISPU_PATH = RAW_DIR / "ispu.csv"
WEATHER_PATH = RAW_DIR / "ispu_weather.csv"
OUTPUT_PATH = PROCESSED_DIR / "hourly_per_sensor.csv"

DEVICE_COL = "ispu_device_id"

POLLUTANT_COLS = ["PM25", "PM10", "CO", "O3", "SO2", "NO2"]

# HEDRO diduga kadar hidrokarbon (HC), karena AQI_KEY sering bernilai "HC"
ISPU_EXTRA_COLS = ["HEDRO", "AQI"]

WEATHER_COLS = ["TMP", "RAIN", "HUM", "WS", "WD", "PR", "UV", "LIGHT", "NOISE"]

# Batas nilai wajar. Nilai di luar batas dianggap error sensor -> NaN.
# Batas atas polutan = konsentrasi pada ISPU 500 (Permen LHK No. 14/2020).
# Periksa kembali dengan tabel resmi sebelum dipakai di laporan.
VALID_RANGES = {
    "PM25": (0, 500),
    "PM10": (0, 500),
    "CO": (0, 45000),
    "O3": (0, 1000),
    "SO2": (0, 1200),
    "NO2": (0, 3000),
    "TMP": (10, 45),
    "HUM": (1, 100),
    "PR": (900, 1100),
    "WS": (0, 30),
    "WD": (0, 360),
    "RAIN": (0, 100),
    "UV": (0, 20),
    "LIGHT": (0, 200000),
    "NOISE": (20, 140),
}

# Minimal jumlah slot 5 menit agar satu jam dianggap valid (12 = penuh)
MIN_SLOTS_PER_HOUR = 6

print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)
print("Input  :", ISPU_PATH.resolve())
print("         ", WEATHER_PATH.resolve())
print("Output :", OUTPUT_PATH.resolve())

assert ISPU_PATH.exists(), f"File tidak ditemukan: {ISPU_PATH}"
assert WEATHER_PATH.exists(), f"File tidak ditemukan: {WEATHER_PATH}"

print("\n✓ [NB00-C01] selesai")

Pandas : 3.0.5
NumPy  : 2.5.3
Input  : C:\Users\ASUS\Documents\KP\forecastispu-forecastclaude\data\raw\ispu.csv
          C:\Users\ASUS\Documents\KP\forecastispu-forecastclaude\data\raw\ispu_weather.csv
Output : C:\Users\ASUS\Documents\KP\forecastispu-forecastclaude\data\processed\hourly_per_sensor.csv

✓ [NB00-C01] selesai


In [2]:
# ============================================================
# [NB00-C02] LOAD TABEL ISPU
# ============================================================

ispu_raw = pd.read_csv(
    ISPU_PATH,
    sep=";",
    na_values=["NULL"],
    parse_dates=["DATETIME"]
)

ispu_raw = ispu_raw.rename(columns={"DATETIME": "datetime_db"})

print("Shape   :", ispu_raw.shape)
print("Periode :", ispu_raw["datetime_db"].min(), "→", ispu_raw["datetime_db"].max())
print("Kolom   :", ispu_raw.columns.tolist())

assert DEVICE_COL in ispu_raw.columns
assert ispu_raw["datetime_db"].notna().all(), "Ada DATETIME yang gagal dibaca"

print("\n✓ [NB00-C02] selesai")

Shape   : (231440, 15)
Periode : 2025-10-07 23:35:00 → 2026-09-21 17:40:02
Kolom   : ['id', 'CO', 'NO2', 'O3', 'SO2', 'PM25', 'PM10', 'datetime_db', 'HEDRO', 'AQI', 'AQI_KEY', 'created_at', 'updated_at', 'ispu_device_id', 'ispu_keterangan']

✓ [NB00-C02] selesai


In [3]:
# ============================================================
# [NB00-C03] LOAD TABEL ISPU_WEATHER
# ============================================================

weather_raw = pd.read_csv(
    WEATHER_PATH,
    sep=";",
    na_values=["NULL"],
    parse_dates=["datetime"]
)

weather_raw = weather_raw.rename(columns={"datetime": "datetime_db"})

print("Shape   :", weather_raw.shape)
print("Periode :", weather_raw["datetime_db"].min(), "→", weather_raw["datetime_db"].max())
print("Kolom   :", weather_raw.columns.tolist())

assert DEVICE_COL in weather_raw.columns
assert weather_raw["datetime_db"].notna().all(), "Ada datetime yang gagal dibaca"

print("\n✓ [NB00-C03] selesai")

Shape   : (170832, 16)
Periode : 2026-04-24 16:50:01 → 2026-09-21 17:40:02
Kolom   : ['id', 'ispu_device_id', 'datetime_db', 'TMP', 'RAIN', 'HUM', 'UV', 'MIN_NOISE', 'MAX_NOISE', 'NOISE', 'WS', 'WD', 'PR', 'LIGHT', 'created_at', 'updated_at']

✓ [NB00-C03] selesai


In [4]:
# ============================================================
# [NB00-C04] PROFIL AWAL KEDUA TABEL
# ============================================================

def profile_table(df, name):
    print(f"=== {name} ===")

    per_device = (
        df.groupby(df[DEVICE_COL].fillna("<TANPA ID>"))["datetime_db"]
        .agg(jumlah="count", mulai="min", sampai="max")
    )
    display(per_device)

    missing = df.isna().sum()
    print("Missing per kolom:")
    print(missing[missing > 0].to_string() if missing.any() else "  (tidak ada)")

    dup = df.dropna(subset=[DEVICE_COL]).duplicated([DEVICE_COL, "datetime_db"]).sum()
    print(f"Duplikat (sensor + waktu): {dup}\n")


profile_table(ispu_raw, "ISPU")
profile_table(weather_raw, "ISPU_WEATHER")

print("✓ [NB00-C04] selesai")

=== ISPU ===


,jumlah,mulai,sampai
ispu_device_id,,,
<TANPA ID>,14291,2025-10-07 23:35:00,2025-12-21 02:40:00
PH10P0001,44577,2026-01-12 17:00:02,2026-09-21 17:40:02
PH14P0002,44313,2026-01-12 17:00:02,2026-09-21 17:35:02
PH14P0003,45947,2026-01-09 07:15:02,2026-09-21 17:40:02
PH14P0004,44024,2026-01-12 17:00:01,2026-09-21 17:40:01
PH20P0001,38288,2026-03-14 01:00:02,2026-09-21 17:40:01


Missing per kolom:
CO                  8813
NO2                 8813
O3                  8813
SO2                 8813
PM25                8813
PM10                8813
HEDRO               8813
AQI                    6
AQI_KEY                6
ispu_device_id     14291
ispu_keterangan    14297
Duplikat (sensor + waktu): 23637

=== ISPU_WEATHER ===


,jumlah,mulai,sampai
ispu_device_id,,,
PH10P0001,33529,2026-04-24 16:50:01,2026-09-21 17:40:02
PH14P0002,33589,2026-04-24 16:50:02,2026-09-21 17:35:02
PH14P0003,34825,2026-04-24 16:50:02,2026-09-21 17:40:02
PH14P0004,34750,2026-04-24 16:50:02,2026-09-21 17:40:01
PH20P0001,34139,2026-04-24 16:50:02,2026-09-21 17:40:01


Missing per kolom:
  (tidak ada)
Duplikat (sensor + waktu): 1527

✓ [NB00-C04] selesai


In [5]:
# ============================================================
# [NB00-C05] CLEANING TABEL ISPU
# ============================================================

ispu = ispu_raw.copy()
cleaning_log = [("Awal", len(ispu))]

# 1. Buang baris tanpa ID sensor (tidak bisa ditentukan milik sensor mana)
ispu = ispu[ispu[DEVICE_COL].notna()]
cleaning_log.append(("Buang tanpa ID sensor", len(ispu)))

# 2. Buang baris yang semua polutannya kosong
ispu = ispu[ispu[POLLUTANT_COLS].notna().any(axis=1)]
cleaning_log.append(("Buang semua polutan kosong", len(ispu)))

# 3. Buang duplikat sensor + waktu (isinya identik)
ispu = ispu.drop_duplicates([DEVICE_COL, "datetime_db"], keep="first")
cleaning_log.append(("Buang duplikat", len(ispu)))

# 4. Samakan label AQI_KEY ("PM2.5" dan "PM25" adalah polutan yang sama)
ispu["AQI_KEY"] = ispu["AQI_KEY"].replace({"PM2.5": "PM25"})

# 5. Ambil kolom yang dipakai
ispu = ispu[[DEVICE_COL, "datetime_db"] + POLLUTANT_COLS + ISPU_EXTRA_COLS + ["AQI_KEY"]]

log_df = pd.DataFrame(cleaning_log, columns=["Tahap", "Jumlah baris"])
log_df["Dibuang"] = -log_df["Jumlah baris"].diff().fillna(0).astype(int)
print(log_df.to_string(index=False))

print("\nAQI_KEY setelah dinormalisasi:")
print(ispu["AQI_KEY"].value_counts(dropna=False).to_string())

assert not ispu.duplicated([DEVICE_COL, "datetime_db"]).any()
assert ispu[DEVICE_COL].notna().all()

print("\n✓ [NB00-C05] selesai")

                     Tahap  Jumlah baris  Dibuang
                      Awal        231440        0
     Buang tanpa ID sensor        217149    14291
Buang semua polutan kosong        208336     8813
            Buang duplikat        192568    15768

AQI_KEY setelah dinormalisasi:
AQI_KEY
PM25    165501
HC       24383
PM10      2226
SO2        324
O3          66
NO2         62
NaN          6

✓ [NB00-C05] selesai


In [6]:
# ============================================================
# [NB00-C06] CLEANING TABEL ISPU_WEATHER
# ============================================================

weather = weather_raw.copy()
n_awal = len(weather)

# 1. Buang duplikat sensor + waktu (isinya identik)
weather = weather.drop_duplicates([DEVICE_COL, "datetime_db"], keep="first")

# 2. Ambil kolom yang dipakai
weather = weather[[DEVICE_COL, "datetime_db"] + WEATHER_COLS]

print(f"Baris awal        : {n_awal}")
print(f"Duplikat dibuang  : {n_awal - len(weather)}")
print(f"Baris akhir       : {len(weather)}")

assert not weather.duplicated([DEVICE_COL, "datetime_db"]).any()

print("\n✓ [NB00-C06] selesai")

Baris awal        : 170832
Duplikat dibuang  : 1527
Baris akhir       : 169305

✓ [NB00-C06] selesai


In [7]:
# ============================================================
# [NB00-C07] NILAI DI LUAR BATAS WAJAR -> NaN
# ============================================================

def apply_valid_ranges(df, table_name):
    df = df.copy()
    report = []

    for col, (low, high) in VALID_RANGES.items():
        if col not in df.columns:
            continue

        invalid = df[col].notna() & ~df[col].between(low, high)
        df.loc[invalid, col] = np.nan

        report.append({
            "tabel": table_name,
            "kolom": col,
            "batas": f"{low} – {high}",
            "dijadikan_NaN": int(invalid.sum()),
        })

    return df, pd.DataFrame(report)


ispu, report_ispu = apply_valid_ranges(ispu, "ispu")
weather, report_weather = apply_valid_ranges(weather, "ispu_weather")

range_report = pd.concat([report_ispu, report_weather], ignore_index=True)
print(range_report.to_string(index=False))

# Baris ISPU yang semua polutannya menjadi NaN setelah filter ikut dibuang
n_before = len(ispu)
ispu = ispu[ispu[POLLUTANT_COLS].notna().any(axis=1)]
print(f"\nBaris ISPU dibuang karena semua polutan tidak valid: {n_before - len(ispu)}")

print("\n✓ [NB00-C07] selesai")

       tabel kolom      batas  dijadikan_NaN
        ispu  PM25    0 – 500             94
        ispu  PM10    0 – 500            615
        ispu    CO  0 – 45000              0
        ispu    O3   0 – 1000              0
        ispu   SO2   0 – 1200             16
        ispu   NO2   0 – 3000              0
ispu_weather   TMP    10 – 45              1
ispu_weather   HUM    1 – 100              1
ispu_weather    PR 900 – 1100          34304
ispu_weather    WS     0 – 30            256
ispu_weather    WD    0 – 360              0
ispu_weather  RAIN    0 – 100              0
ispu_weather    UV     0 – 20              0
ispu_weather LIGHT 0 – 200000              0
ispu_weather NOISE   20 – 140            873

Baris ISPU dibuang karena semua polutan tidak valid: 0

✓ [NB00-C07] selesai


In [8]:
# ============================================================
# [NB00-C08] BULATKAN WAKTU KE SLOT 5 MENIT
# ============================================================
# Timestamp memiliki detik berbeda (:01, :02), sehingga dibulatkan ke 5 menit
# agar kedua tabel bisa dipasangkan. Jika satu sensor memiliki lebih dari satu
# catatan di slot yang sama, nilainya dirata-rata.

# Arah angin diubah ke sin/cos sebelum dirata-rata.
# Rata-rata langsung dalam derajat salah: rata-rata 350° dan 10° menjadi 180°.
weather["WD_sin"] = np.sin(np.deg2rad(weather["WD"]))
weather["WD_cos"] = np.cos(np.deg2rad(weather["WD"]))
weather = weather.drop(columns="WD")

WEATHER_VALUE_COLS = [c for c in WEATHER_COLS if c != "WD"] + ["WD_sin", "WD_cos"]


def to_5min_slots(df, value_cols, extra_agg=None):
    df = df.copy()
    df["slot_db"] = df["datetime_db"].dt.floor("5min")

    agg = {col: "mean" for col in value_cols}
    if extra_agg:
        agg.update(extra_agg)

    out = df.groupby([DEVICE_COL, "slot_db"]).agg(agg).reset_index()
    return out


ispu_5 = to_5min_slots(
    ispu,
    POLLUTANT_COLS + ISPU_EXTRA_COLS,
    extra_agg={"AQI_KEY": "first"}
)

weather_5 = to_5min_slots(weather, WEATHER_VALUE_COLS)

print(f"ISPU    : {len(ispu):>7} baris → {len(ispu_5):>7} slot  "
      f"(digabung: {len(ispu) - len(ispu_5)})")
print(f"WEATHER : {len(weather):>7} baris → {len(weather_5):>7} slot  "
      f"(digabung: {len(weather) - len(weather_5)})")

assert not ispu_5.duplicated([DEVICE_COL, "slot_db"]).any()
assert not weather_5.duplicated([DEVICE_COL, "slot_db"]).any()

print("\n✓ [NB00-C08] selesai")

ISPU    :  192568 baris →  192566 slot  (digabung: 2)
WEATHER :  169305 baris →  169305 slot  (digabung: 0)

✓ [NB00-C08] selesai


In [9]:
# ============================================================
# [NB00-C09] MERGE ISPU + WEATHER (SENSOR + SLOT 5 MENIT)
# ============================================================
# - Kunci merge: ispu_device_id + slot_db
# - validate="one_to_one" -> error jika ada duplikasi (masalah di file merge lama)
# - how="left" -> semua data polutan (target) dipertahankan;
#   slot tanpa pasangan cuaca akan berisi NaN di kolom cuaca

merged = ispu_5.merge(
    weather_5,
    on=[DEVICE_COL, "slot_db"],
    how="left",
    validate="one_to_one",
    indicator=True
)

merged["has_weather"] = merged["_merge"] == "both"
merged = merged.drop(columns="_merge")

print(f"Slot ISPU               : {len(ispu_5)}")
print(f"Slot hasil merge        : {len(merged)}")
print(f"Punya pasangan cuaca    : {merged['has_weather'].sum()} "
      f"({merged['has_weather'].mean():.1%})")

weather_only = len(weather_5) - merged["has_weather"].sum()
print(f"Slot cuaca tanpa ISPU   : {weather_only} (tidak dipakai, karena tidak ada target)")

# Jumlah baris tidak boleh bertambah setelah merge
assert len(merged) == len(ispu_5), "Jumlah baris berubah setelah merge!"

print("\n✓ [NB00-C09] selesai")

Slot ISPU               : 192566
Slot hasil merge        : 192566
Punya pasangan cuaca    : 167515 (87.0%)
Slot cuaca tanpa ISPU   : 1790 (tidak dipakai, karena tidak ada target)

✓ [NB00-C09] selesai


In [10]:
# ============================================================
# [NB00-C10] CEK CAKUPAN MERGE PER SENSOR
# ============================================================

weather_start = weather_5["slot_db"].min()

coverage = (
    merged.assign(setelah_cuaca_mulai=merged["slot_db"] >= weather_start)
    .groupby(DEVICE_COL)
    .agg(
        slot_ispu=("slot_db", "count"),
        mulai=("slot_db", "min"),
        sampai=("slot_db", "max"),
        slot_sejak_cuaca_mulai=("setelah_cuaca_mulai", "sum"),
        punya_cuaca=("has_weather", "sum"),
    )
)

coverage["cakupan_cuaca_sejak_mulai"] = (
    coverage["punya_cuaca"] / coverage["slot_sejak_cuaca_mulai"]
).map("{:.1%}".format)

print(f"Data cuaca mulai tersedia: {weather_start}\n")
display(coverage)

print("✓ [NB00-C10] selesai")

Data cuaca mulai tersedia: 2026-04-24 16:50:00



,slot_ispu,mulai,sampai,slot_sejak_cuaca_mulai,punya_cuaca,cakupan_cuaca_sejak_mulai
ispu_device_id,,,,,,
PH10P0001,38231,2026-01-12 17:00:00,2026-09-21 17:40:00,36847,33373,90.6%
PH14P0002,37942,2026-01-12 17:00:00,2026-09-21 17:35:00,36584,32363,88.5%
PH14P0003,39596,2026-02-02 15:50:00,2026-09-21 17:40:00,38218,34672,90.7%
PH14P0004,38518,2026-01-12 17:00:00,2026-09-21 17:40:00,36295,33344,91.9%
PH20P0001,38279,2026-03-14 01:00:00,2026-09-21 17:40:00,37374,33763,90.3%


✓ [NB00-C10] selesai


In [11]:
# ============================================================
# [NB00-C11] AGREGASI PER JAM PER SENSOR
# ============================================================
# - Label jam = awal jam (contoh: 02:00 = data 02:00–02:55)
# - Jam dianggap valid jika minimal MIN_SLOTS_PER_HOUR slot 5 menit tersedia
# - Arah angin dikembalikan ke derajat dari rata-rata sin/cos

merged["hour_db"] = merged["slot_db"].dt.floor("1h")
merged["ispu_ok"] = merged[POLLUTANT_COLS].notna().any(axis=1)

mean_cols = POLLUTANT_COLS + ISPU_EXTRA_COLS + WEATHER_VALUE_COLS

hourly = (
    merged.groupby([DEVICE_COL, "hour_db"])
    .agg(
        **{col: (col, "mean") for col in mean_cols},
        n_slots_ispu=("ispu_ok", "sum"),
        n_slots_weather=("has_weather", "sum"),
    )
    .reset_index()
)

hourly["WD"] = np.rad2deg(np.arctan2(hourly["WD_sin"], hourly["WD_cos"])) % 360

# Terapkan batas minimal slot per jam
low_ispu = hourly["n_slots_ispu"] < MIN_SLOTS_PER_HOUR
low_weather = hourly["n_slots_weather"] < MIN_SLOTS_PER_HOUR

hourly.loc[low_ispu, POLLUTANT_COLS + ISPU_EXTRA_COLS] = np.nan
hourly.loc[low_weather, WEATHER_VALUE_COLS + ["WD"]] = np.nan

print(f"Jumlah baris per jam       : {len(hourly)}")
print(f"Jam polutan tidak valid    : {low_ispu.sum()} (slot < {MIN_SLOTS_PER_HOUR})")
print(f"Jam cuaca tidak valid      : {low_weather.sum()} (slot < {MIN_SLOTS_PER_HOUR})")

assert not hourly.duplicated([DEVICE_COL, "hour_db"]).any()

print("\n✓ [NB00-C11] selesai")

Jumlah baris per jam       : 23217
Jam polutan tidak valid    : 7430 (slot < 6)
Jam cuaca tidak valid      : 8081 (slot < 6)

✓ [NB00-C11] selesai


In [12]:
# ============================================================
# [NB00-C12] LENGKAPI TIMELINE PER JAM & CEK GAP PER SENSOR
# ============================================================
# Jam yang tidak punya data sama sekali dimasukkan sebagai baris NaN,
# sehingga timeline setiap sensor lengkap per jam.
# Gap TIDAK diisi (tidak diinterpolasi) agar tidak ada kebocoran data
# dari masa depan. Penanganan NaN dilakukan di NB03.

full_parts = []
gap_report = []

for device, df_dev in hourly.groupby(DEVICE_COL):
    df_dev = df_dev.set_index("hour_db").sort_index()

    full_index = pd.date_range(df_dev.index.min(), df_dev.index.max(), freq="1h", name="hour_db")
    df_dev = df_dev.reindex(full_index)
    df_dev[DEVICE_COL] = device
    df_dev[["n_slots_ispu", "n_slots_weather"]] = (
        df_dev[["n_slots_ispu", "n_slots_weather"]].fillna(0).astype(int)
    )

    # Panjang gap terpanjang (jam berturut-turut tanpa data PM25)
    missing = df_dev["PM25"].isna()
    run_id = missing.ne(missing.shift()).cumsum()
    longest_gap = missing.groupby(run_id).sum().max()

    gap_report.append({
        "sensor": device,
        "total_jam": len(df_dev),
        "jam_polutan_valid": int(df_dev["PM25"].notna().sum()),
        "jam_cuaca_valid": int(df_dev["TMP"].notna().sum()),
        "jam_lengkap": int(df_dev[["PM25", "TMP"]].notna().all(axis=1).sum()),
        "gap_terpanjang_jam": int(longest_gap),
    })

    full_parts.append(df_dev.reset_index())

hourly_full = pd.concat(full_parts, ignore_index=True)

gap_df = pd.DataFrame(gap_report)
gap_df["persen_lengkap"] = (gap_df["jam_lengkap"] / gap_df["total_jam"]).map("{:.1%}".format)
print(gap_df.to_string(index=False))

assert hourly_full[DEVICE_COL].nunique() == hourly[DEVICE_COL].nunique()
assert not hourly_full.duplicated([DEVICE_COL, "hour_db"]).any()

print("\n✓ [NB00-C12] selesai")

   sensor  total_jam  jam_polutan_valid  jam_cuaca_valid  jam_lengkap  gap_terpanjang_jam persen_lengkap
PH10P0001       6049               3119             2989         2989                2447          49.4%
PH14P0002       6049               3203             3050         3049                2447          50.4%
PH14P0003       5547               3234             3097         3097                1945          55.8%
PH14P0004       6049               3069             2976         2976                2447          49.2%
PH20P0001       4601               3161             3024         3024                 999          65.7%

✓ [NB00-C12] selesai


In [13]:
# ============================================================
# [NB00-C13] SIMPAN DATA PER JAM PER SENSOR
# ============================================================

output_cols = (
    [DEVICE_COL, "hour_db"]
    + POLLUTANT_COLS
    + ISPU_EXTRA_COLS
    + ["TMP", "RAIN", "HUM", "WS", "WD", "WD_sin", "WD_cos", "PR", "UV", "LIGHT", "NOISE"]
    + ["n_slots_ispu", "n_slots_weather"]
)

hourly_out = (
    hourly_full[output_cols]
    .rename(columns={"hour_db": "datetime_db"})
    .sort_values([DEVICE_COL, "datetime_db"])
    .reset_index(drop=True)
)

hourly_out.to_csv(OUTPUT_PATH, index=False)

print("File disimpan :", OUTPUT_PATH.resolve())
print("Shape         :", hourly_out.shape)
print("Sensor        :", hourly_out[DEVICE_COL].unique().tolist())
print("Periode       :", hourly_out["datetime_db"].min(), "→", hourly_out["datetime_db"].max())

# Verifikasi: file bisa dibaca ulang dengan isi yang sama
check = pd.read_csv(OUTPUT_PATH, parse_dates=["datetime_db"])
assert check.shape == hourly_out.shape
assert not check.duplicated([DEVICE_COL, "datetime_db"]).any()

display(hourly_out.head())

print("\n✓ [NB00-C13] selesai")

File disimpan : C:\Users\ASUS\Documents\KP\forecastispu-forecastclaude\data\processed\hourly_per_sensor.csv
Shape         : (28295, 23)
Sensor        : ['PH10P0001', 'PH14P0002', 'PH14P0003', 'PH14P0004', 'PH20P0001']
Periode       : 2026-01-12 17:00:00 → 2026-09-21 17:00:00


,ispu_device_id,datetime_db,PM25,PM10,CO,O3,SO2,NO2,HEDRO,AQI,...,WS,WD,WD_sin,WD_cos,PR,UV,LIGHT,NOISE,n_slots_ispu,n_slots_weather
0,PH10P0001,2026-01-12 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
1,PH10P0001,2026-01-12 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,PH10P0001,2026-01-12 19:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
3,PH10P0001,2026-01-12 20:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
4,PH10P0001,2026-01-12 21:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0



✓ [NB00-C13] selesai
